In [31]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Customer_support_chatbot'

!pip install -q datasets sentence-transformers faiss-cpu groq

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from datasets import load_dataset

support_ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = support_ds['train'].to_pandas()
df.head()

#care about two columns:
#instruction (a customer's past question)
#response (the correct agent answer to it).
#Together they're your "knowledge base"

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


turn every past question into an embedding

In [23]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
instructions = df['instruction'].tolist()

embeddings = embedder.encode(instructions,
                             show_progress_bar=True,
                             batch_size=64)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/420 [00:00<?, ?it/s]

(26872, 384)


build FAISS index `'a searchable structure for thos vectors'`

In [24]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype('float32')
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Total vectors in index:", index.ntotal)


Total vectors in index: 26872


setup Groq for the LLM step

In [25]:
from groq import Groq
import getpass

groq_api = getpass.getpass("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")
client = Groq(api_key=groq_api)

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··········


The retrieval function (find the most relevant past Q&As)

In [26]:
'''asks FAISS for the top_k (here, 3)
closest matches from your filing cabinet.
It returns the actual question+answer pairs, not just numbers'''
def retrieve(query, top_k=3):
  query_vec = embedder.encode([query]).astype('float32')
  distances, indices = index.search(query_vec, top_k)
  return []
  for idx in indices[0]:
    results.append({
        'instruction': df.iloc[idx]['instruction'],
        'response': df.iloc[idx]['response']
    })
    return results

full rag function (retrieve + generate)

In [27]:
def rag_answer(user_message, detected_sentiment="neutral"):
    retrieved = retrieve(user_message, top_k=3)

    context_text = "\n\n".join([f"Q: {r['instruction']}\nA: {r['response']}" for r in retrieved])

    system_prompt = f"""You are a helpful, professional customer support assistant for an online retailer.
Answer the customer's question using ONLY the information in the retrieved support responses below.
If the customer sounds frustrated ({detected_sentiment}), acknowledge that before answering.
If the retrieved context does not cover the question, say so honestly and offer to escalate to a human agent rather than guessing."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context_text}\n\nCustomer question: \"{user_message}\""}
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content

In [28]:
print(rag_answer("Where is my order? It's been a week!\n", detected_sentiment="negative"))
print(rag_answer("What is your refund policy?", detected_sentiment="neutral"))

I’m sorry you’re feeling frustrated—let’s get this sorted out. Unfortunately, I don’t have any specific information about your order status in the data I’ve been given. I can’t answer that question right now. I’ll forward your request to a human agent who can look into the details and get you an update as soon as possible. Thank you for your patience.
I’m sorry I don’t have the refund policy details in front of me right now. I can connect you with one of our human agents who can give you the full information. Would you like me to do that?


In [32]:
import os
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)

faiss.write_index(index, f'{PROJECT_DIR}/models/support_faiss.index')
df.to_pickle(f'{PROJECT_DIR}/models/support_df.pkl')